In [137]:
!pip install rdkit
!pip install torch torch-geometric rdkit pandas numpy scikit-learn



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [138]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import *


In [139]:

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import *

from torch.utils.data import TensorDataset, DataLoader

In [140]:
import pandas as pd

# file path
fn = r"C:\Users\ASUS\Downloads\WholedatasetGAN.csv"

# load dataset
df = pd.read_csv(fn)

# ----- Binary classification label from pIC50 -----
# active = 1 if pIC50 >= 6
df["active"] = (df["pIC50"] >= 6).astype(int)

# preview
df.head()


,SMILES,pIC50,active
0,C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=...,3.537602,0
1,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H...,3.677781,0
2,C1=CC=C(C(=C1)C(=O)C2=CN=C(S2)NC3=CC=C(C=C3)N)Cl,3.698970,0
3,C1CCC(=O)C(C1)C2CC(=O)N(C2=O)C3=CC=C(C=C3)Br,3.884389,0
4,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H...,3.920819,0


In [141]:
from rdkit.Chem import AllChem, DataStructs

def calculate_tanimoto_diversity(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols]
    
    similarities = []
    # Sample 500 molecules if dataset is large to save time
    sample_size = min(len(fps), 500) 
    for i in range(sample_size):
        for j in range(i + 1, sample_size):
            similarities.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
            
    avg_sim = np.mean(similarities)
    print(f"Average Tanimoto Similarity: {avg_sim:.4f}")
    return avg_sim

# Run analysis on your anti-inflammatory dataset
calculate_tanimoto_diversity(df['SMILES'].tolist())

[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerator
[13:47:06] DEPRECATION WARNING: please use MorganGenerat

Average Tanimoto Similarity: 0.1182


np.float64(0.11815925451287997)

In [142]:
df = df.dropna(subset=["pIC50"])


In [143]:
df["active"].value_counts()


active
0    2150
1    2150
Name: count, dtype: int64

In [144]:
df[["pIC50","active"]].head(10)


,pIC50,active
0,3.537602,0
1,3.677781,0
2,3.698970,0
3,3.884389,0
4,3.920819,0
5,3.935542,0
6,4.000000,0
7,4.000000,0
8,4.000000,0
9,4.000000,0


In [145]:
atom_set = set()

for smi in df.SMILES:
    m = Chem.MolFromSmiles(smi)
    if m:
        for a in m.GetAtoms():
            atom_set.add(a.GetSymbol())

ATOM_TYPES = sorted(atom_set)

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC
]

MAX_ATOMS = 45

print("Atom types:", ATOM_TYPES)


Atom types: ['Br', 'C', 'Cl', 'F', 'H', 'I', 'N', 'Na', 'O', 'P', 'S']


In [146]:
# Scan for all bond types in your dataset
bond_set = set()
for smi in df.SMILES:
    m = Chem.MolFromSmiles(smi)
    if m:
        for b in m.GetBonds():
            bond_set.add(b.GetBondType())
print("Bond types found in dataset:", bond_set)

# Ensure BOND_TYPES matches the diversity found
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC
]

Bond types found in dataset: {rdkit.Chem.rdchem.BondType.SINGLE, rdkit.Chem.rdchem.BondType.DOUBLE, rdkit.Chem.rdchem.BondType.TRIPLE, rdkit.Chem.rdchem.BondType.AROMATIC}


In [147]:
def atom_features(atom):
    # Existing one-hot for atom types
    onehot = [int(atom.GetSymbol() == a) for a in ATOM_TYPES]
    # Advanced features for anti-inflammatory structural awareness
    extra = [
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()), # Crucial for identifying bioactive rings
        int(atom.GetHybridization()) # Captures sp2/sp3 geometry
    ]
    return onehot + extra + [int(sum(onehot) == 0)]


In [148]:
def build_X(m):

    Fdim = len(atom_features(m.GetAtomWithIdx(0)))
    X = np.zeros((MAX_ATOMS,Fdim),np.float32)

    for i,a in enumerate(m.GetAtoms()):
        if i<MAX_ATOMS:
            X[i] = atom_features(a)

    return X


In [149]:
def build_A(m):

    A = np.zeros((MAX_ATOMS,MAX_ATOMS,len(BOND_TYPES)+1),np.float32)

    for b in m.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if i<MAX_ATOMS and j<MAX_ATOMS:
            bt = b.GetBondType()
            if bt in BOND_TYPES:
                k = BOND_TYPES.index(bt)+1
                A[i,j,k]=1
                A[j,i,k]=1

    A[:,:,0] = (A.sum(2)==0)
    return A


In [150]:
Xs,As,yc,yr = [],[],[],[]

for smi,a,p in zip(df.SMILES,df.active,df.pIC50):

    m = Chem.MolFromSmiles(smi)

    if m:
        Xs.append(build_X(m))
        As.append(build_A(m))
        yc.append(a)
        yr.append(p)

X = torch.tensor(np.array(Xs)).float()
A = torch.tensor(np.array(As)).float()
yc = torch.tensor(yc).float()
yr = torch.tensor(yr).float()

print(X.shape,A.shape)


torch.Size([4300, 45, 18]) torch.Size([4300, 45, 45, 5])


In [151]:
from rdkit.Chem.Scaffolds import MurckoScaffold

# 1. Identify Scaffolds for every molecule to prevent 'cheating'
df['scaffold'] = df['SMILES'].apply(lambda x: MurckoScaffold.MurckoScaffoldSmiles(smiles=x, includeChirality=True))

# 2. Get unique scaffolds and shuffle them
unique_scaffolds = df['scaffold'].unique()
np.random.seed(42)
np.random.shuffle(unique_scaffolds)

# 3. Determine the split point (80/20)
train_size = int(0.8 * len(unique_scaffolds))
train_scaffolds = unique_scaffolds[:train_size]

# 4. Map back to original indices
train_idx = df[df['scaffold'].isin(train_scaffolds)].index.values
test_idx = df[~df['scaffold'].isin(train_scaffolds)].index.values

# 5. FIXED: Assign tensors to variables so they can be standardized in the next step
# This ensures 'yr_train' and others are defined for your math cell
X_train, A_train = X[train_idx], A[train_idx]
yc_train, yr_train = yc[train_idx], yr[train_idx]

X_test, A_test = X[test_idx], A[test_idx]
yc_test, yr_test = yc[test_idx], yr[test_idx]

print(f"Scaffold Split - Training: {len(train_idx)} | Test: {len(test_idx)}")

Scaffold Split - Training: 3469 | Test: 831


In [152]:
yr_mean = yr_train.mean()
yr_std  = yr_train.std()+1e-8

yr_train = (yr_train-yr_mean)/yr_std
yr_test  = (yr_test-yr_mean)/yr_std


In [153]:
def cls_metrics(y,p):

    yhat=(p>=0.5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,yhat).ravel()

    return {
        "ACC":accuracy_score(y,yhat),
        "MCC":matthews_corrcoef(y,yhat),
        "AUROC":roc_auc_score(y,p),
        "Sensitivity":tp/(tp+fn+1e-8),
        "Specificity":tn/(tn+fp+1e-8)
    }

def reg_metrics(y,p):

    return {
        "RMSE":np.sqrt(mean_squared_error(y,p)),
        "MAE":mean_absolute_error(y,p),
        "R2":r2_score(y,p)
    }


In [154]:
class GCNLayer(nn.Module):

    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim,out_dim)

    def forward(self,X,A):

        A_bin = (A[:,:,:,1:].sum(dim=3)>0).float()

        I = torch.eye(A_bin.size(1),device=A_bin.device)
        A_hat = A_bin + I

        D = A_hat.sum(dim=2)
        D_inv = torch.pow(D+1e-6,-0.5)
        D_inv = torch.diag_embed(D_inv)

        A_norm = D_inv @ A_hat @ D_inv

        return F.relu(self.lin(A_norm @ X))


In [155]:
class GATLayer(nn.Module):

    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim,out_dim)
        self.att = nn.Linear(out_dim*2,1)

    def forward(self,X,A):

        H = self.W(X)
        A_bin = (A[:,:,:,1:].sum(dim=3)>0)

        B,N,D = H.shape   # ← FIXED (was F before)

        Hi = H.unsqueeze(2).repeat(1,1,N,1)
        Hj = H.unsqueeze(1).repeat(1,N,1,1)

        e = self.att(torch.cat([Hi,Hj],3)).squeeze(-1)
        e = e.masked_fill(~A_bin,-1e9)

        alpha = torch.softmax(e,2)

        return F.relu(alpha @ H)


In [156]:
class GTLayer(nn.Module):

    def __init__(self,dim):
        super().__init__()

        self.att = nn.MultiheadAttention(dim,4,batch_first=True)

        self.ffn = nn.Sequential(
            nn.Linear(dim,dim*2),
            nn.ReLU(),
            nn.Linear(dim*2,dim)
        )

        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self,X):

        h,_ = self.att(X,X,X)
        X = self.ln1(X+h)

        h2 = self.ffn(X)
        return self.ln2(X+h2)


In [157]:
in_dim = X.shape[2]
H = 128


In [158]:
class GCNModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.g1 = GCNLayer(in_dim,H)
        self.g2 = GCNLayer(H,H)

        self.cls = nn.Linear(H,1)
        self.reg = nn.Linear(H,1)

    def forward(self,X,A):

        h = self.g2(self.g1(X,A),A).mean(1)

        return self.cls(h).view(-1), self.reg(h).view(-1)


In [159]:
class GATModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.g1 = GATLayer(in_dim,H)
        self.g2 = GATLayer(H,H)

        self.cls = nn.Linear(H,1)
        self.reg = nn.Linear(H,1)

    def forward(self,X,A):

        h = self.g2(self.g1(X,A),A).mean(1)

        return self.cls(h).view(-1), self.reg(h).view(-1)


In [160]:
class GTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.emb = nn.Linear(in_dim,H)
        self.t1 = GTLayer(H)
        self.t2 = GTLayer(H)

        self.cls = nn.Linear(H,1)
        self.reg = nn.Linear(H,1)

    def forward(self,X,A):

        h = self.emb(X)
        h = self.t2(self.t1(h)).mean(1)

        return self.cls(h).view(-1), self.reg(h).view(-1)


In [161]:
class FusionGraphModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.gcn = GCNModel()
        self.gat = GATModel()

        self.fuse = nn.Linear(2,H)

        self.cls = nn.Linear(H,1)
        self.reg = nn.Linear(H,1)

    def forward(self,X,A):

        c1,r1 = self.gcn(X,A)
        c2,r2 = self.gat(X,A)

        h = torch.stack([c1,c2],1)
        h = F.relu(self.fuse(h))

        return self.cls(h).view(-1), self.reg(h).view(-1)


In [162]:
class MSMP_Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.gcn1 = GCNLayer(in_dim,H)
        self.gat1 = GATLayer(in_dim,H)

        self.emb = nn.Linear(in_dim,H)
        self.gt = GTLayer(H)

        self.fuse = nn.Linear(H*3,H)

        self.cls = nn.Linear(H,1)
        self.reg = nn.Linear(H,1)
        self.dropout = nn.Dropout(0.2)

    def forward(self,X,A):

        h1 = self.gcn1(X,A).mean(1)
        h2 = self.gat1(X,A).mean(1)

        h3 = self.gt(self.emb(X)).mean(1)

        h = torch.cat([h1, h2, h3], 1)
        h = self.dropout(F.relu(self.fuse(h))) # Apply dropout
        return self.cls(h).view(-1), self.reg(h).view(-1)


In [163]:
def train_model_with_early_stopping(ModelClass, X_train, A_train, yc_train, yr_train):
    model = ModelClass()
    
    # Advanced Optimizer with weight decay for better structural generalization
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
    
    # 1. NEW: Learning Rate Scheduler 
    # This automatically reduces the LR when the model stops improving, 
    # allowing the AI to "fine-tune" its way to a higher R2
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    
    bc = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()
    
    best_loss = float('inf')
    patience_limit = 10
    counter = 0

    loader = DataLoader(
        TensorDataset(X_train, A_train, yc_train, yr_train), 
        batch_size=32, 
        shuffle=True
    )
    
    for epoch in range(120):
        model.train()
        epoch_loss = 0
        
        for Xb, Ab, yb, yrb in loader:
            lc, lrp = model(Xb, Ab)
            
            # Weighted Multi-Task Loss
            loss = 0.5 * bc(lc, yb) + 0.5 * mse(lrp, yrb)
            
            optimizer.zero_grad()
            loss.backward()
            
            # 2. NEW: Gradient Clipping
            # Prevents mathematical instability in the Attention and Transformer layers
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(loader)
        scheduler.step(avg_loss) # Adjusts LR based on current fold performance
            
        if avg_loss < best_loss:
            best_loss = avg_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience_limit:
                print(f"Early stopping at epoch {epoch}")
                break
                
    return model

In [164]:
def test_eval_custom(model, X_te, A_te, yc_te, yr_te, y_mean, y_std):
    model.eval()
    P, R = [], []
    with torch.no_grad():
        # Uses the specific test fold data provided by the CV loop
        for Xb, Ab in DataLoader(TensorDataset(X_te, A_te), 128):
            lc, lrp = model(Xb, Ab)
            P.append(torch.sigmoid(lc))
            R.append(lrp)
            
    P = torch.cat(P).cpu().numpy()
    R = torch.cat(R).cpu()
    
    # Standardizes based on the current fold's mean/std
    R = (R * y_std + y_mean).numpy()
    yc_np = yc_te.cpu().numpy()
    yr_np = yr_te.cpu().numpy()
    
    return cls_metrics(yc_np, P), reg_metrics(yr_np, R)

In [165]:
from sklearn.model_selection import KFold

def run_5_fold_vc(ModelClass):
    # Fulfills "Statistical Evaluation" requirement for mean ± std
    unique_scaffs = df['scaffold'].unique()
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    fold_results_cls = []
    fold_results_reg = []
    
    for fold, (train_scaff_idx, test_scaff_idx) in enumerate(kf.split(unique_scaffs)):
        # 1. Identify scaffolds for this specific fold
        train_scaffs = unique_scaffs[train_scaff_idx]
        t_idx = df[df['scaffold'].isin(train_scaffs)].index.values
        v_idx = df[~df['scaffold'].isin(train_scaffs)].index.values
        
        # 2. Extract fold-specific data from global tensors
        X_tr, A_tr, yc_tr, yr_tr = X[t_idx], A[t_idx], yc[t_idx], yr[t_idx]
        X_te, A_te, yc_te, yr_te = X[v_idx], A[v_idx], yc[v_idx], yr[v_idx]
        
        # 3. Standardization (Must be fold-specific to prevent data leakage)
        y_mean, y_std = yr_tr.mean(), yr_tr.std() + 1e-8
        yr_tr_norm = (yr_tr - y_mean) / y_std
        yr_te_norm = (yr_te - y_mean) / y_std
        
        # 4. Train with optimizations (Scheduler + Clipping)
        model = train_model_with_early_stopping(ModelClass, X_tr, A_tr, yc_tr, yr_tr_norm)
        
        # 5. Evaluate
        c_met, r_met = test_eval_custom(model, X_te, A_te, yc_te, yr_te, y_mean, y_std)
        
        fold_results_cls.append(c_met)
        fold_results_reg.append(r_met)
        print(f"Fold {fold+1} Complete: ACC {c_met['ACC']:.3f}, R2 {r_met['R2']:.3f}")

    # Final reporting for your research paper
    print(f"\nFINAL RESULTS FOR {ModelClass.__name__}:")
    for metric in ['ACC', 'MCC', 'AUROC']:
        vals = [f[metric] for f in fold_results_cls]
        print(f"{metric}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")

In [166]:
# Run the 5-fold cross-validation for your best model
run_5_fold_vc(MSMP_Model)

Fold 1 Complete: ACC 0.724, R2 0.411
Fold 2 Complete: ACC 0.729, R2 0.387
Fold 3 Complete: ACC 0.740, R2 0.384
Fold 4 Complete: ACC 0.800, R2 0.492
Fold 5 Complete: ACC 0.757, R2 0.399

FINAL RESULTS FOR MSMP_Model:
ACC: 0.750 ± 0.028
MCC: 0.500 ± 0.055
AUROC: 0.837 ± 0.031
